# Chapitre 2 · Les nombres qui apprennent (solutions des exercices)

Ce notebook contient **uniquement les réponses aux trois exercices** du
notebook du chapitre. Le code de la leçon, lui, vit dans le notebook du
chapitre et dans le livre.

Si tu n'as pas encore vraiment essayé les exercices, referme ceci : le pacte
« IA débranchée » vaut aussi pour les corrigés.

In [ ]:
# Mise en place (reprise de la leçon) : le minimum pour que tout s'exécute ici.
import torch

doses = torch.tensor([3.0, 2.0])       # la recette : 3 ananas, 2 poignées de gingembre
prix  = torch.tensor([250.0, 100.0])   # les prix à Dantokpa, en FCFA
D = torch.tensor([[3.0, 2.0],
                  [1.0, 4.0]])         # doses : 2 recettes x 2 ingrédients
P = torch.tensor([[250.0, 400.0],
                  [100.0, 150.0]])     # prix : 2 ingrédients x 2 fournisseurs

### Exercice 1 · Produit scalaire et matmul, version PyTorch — niveau ●

La version à trois boucles de la leçon, tu ne l'écriras plus jamais : PyTorch
fait la même chose en beaucoup plus rapide. `torch.dot` pour le produit
scalaire, l'opérateur `@` pour le produit matriciel. Refais les deux calculs
d'Ayélé avec les outils de tous les jours.

In [ ]:
cout_bidon = torch.dot(doses, prix)    # le produit scalaire, en un geste
couts_torch = D @ P                     # le produit matriciel, en une ligne

print(cout_bidon)      # attendu : tensor(950.)
print(couts_torch)     # attendu : [[ 950., 1500.], [ 650., 1000.]]

In [ ]:
# Validation : la main et la machine doivent dire exactement la même chose.
attendu = torch.tensor([[950.0, 1500.0],
                        [650.0, 1000.0]])
assert float(cout_bidon) == 950.0, "torch.dot(doses, prix) doit valoir 950"
assert torch.allclose(couts_torch, attendu), "D @ P doit donner les quatre coûts"

# Au passage : * seul ne fait QUE la première moitié du geste.
assert torch.allclose(doses * prix, torch.tensor([750.0, 200.0]))
assert float((doses * prix).sum()) == 950.0    # multiplié PUIS sommé = produit scalaire

print("Vérification OK : @ fabrique bien une collection de produits scalaires.")

### Exercice 2 · Le produit scalaire à la main — niveau ●●

Réécris `produit_scalaire`, sans regarder la leçon : multiplier les deux
vecteurs **position par position**, puis **tout additionner**, avec une simple
boucle Python. Interdit ici : `torch.dot`, `@`, `(u * v).sum()`. C'est toi le
processeur.

In [ ]:
def produit_scalaire(u, v):
    assert u.shape == v.shape, "deux vecteurs de même longueur"
    total = 0.0
    for i in range(len(u)):              # pour chaque position...
        total += float(u[i]) * float(v[i])   # ...multiplier, puis accumuler
    return total


print(produit_scalaire(doses, prix))    # attendu : 950.0

In [ ]:
# Validation du produit scalaire : le coût du bidon, puis les cas purs.
assert abs(produit_scalaire(doses, prix) - 950.0) < 1e-6, (
    "doses . prix doit valoir 950 FCFA (3 x 250 + 2 x 100)"
)

nouvelle = torch.tensor([2.0, 3.0])      # 2 ananas, 3 poignées de gingembre
assert abs(produit_scalaire(nouvelle, prix) - 800.0) < 1e-6, (
    "2 x 250 + 3 x 100 = 800 FCFA"
)

droite = torch.tensor([1.0, 0.0])
haut   = torch.tensor([0.0, 1.0])
gauche = torch.tensor([-1.0, 0.0])
assert produit_scalaire(droite, droite) == 1.0,  "alignées : grand et positif"
assert produit_scalaire(droite, haut)   == 0.0,  "perpendiculaires : zéro"
assert produit_scalaire(droite, gauche) == -1.0, "opposées : négatif"

print("Produit scalaire OK : le bidon coûte 950 FCFA, et les flèches parlent.")

### Exercice 3 · Le produit matriciel à la main — niveau ●●●

La règle tient en une phrase : **la case (ligne i, colonne j) du résultat est
le produit scalaire de la ligne i de la première matrice par la colonne j de
la seconde.** Écris les trois boucles, sans regarder la leçon : c'est le calcul
le plus important du livre, et tes doigts doivent connaître le geste.

In [ ]:
def matmul_a_la_main(A, B):
    m, n = A.shape                    # A : m lignes, n colonnes
    n2, p = B.shape                   # B : n2 lignes, p colonnes
    assert n == n2, "les dimensions du milieu doivent se rencontrer"
    C = torch.zeros(m, p)             # le résultat, prêt à remplir
    for i in range(m):                # pour chaque ligne de A...
        for j in range(p):            # ...et chaque colonne de B...
            for k in range(n):        # ...un produit scalaire complet
                C[i, j] += A[i, k] * B[k, j]
    return C


print(matmul_a_la_main(D, P))         # attendu : [[ 950., 1500.], [ 650., 1000.]]

In [ ]:
# Validation du produit matriciel : les quatre coûts d'Ayélé.
attendu = torch.tensor([[950.0, 1500.0],
                        [650.0, 1000.0]])
couts = matmul_a_la_main(D, P)
assert couts.shape == (2, 2), f"shape attendue (2, 2), obtenue {tuple(couts.shape)}"
assert torch.allclose(couts, attendu), (
    "chaque case (i, j) = produit scalaire de la ligne i de D par la colonne j de P"
)
assert torch.allclose(couts, D @ P), "@ et tes trois boucles font le même calcul"
print("Produit matriciel OK : 950, 1500, 650 et 1000 FCFA, comme au marché.")